In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df_09_21 = pd.read_csv('CollegeBasketballPlayers2009-2021.csv', low_memory=False)

## Rename Unnamed 64 to role

In [3]:
df_09_21 = df_09_21.rename(columns={'Unnamed: 64': 'role'})

## Remove Unnamed 65 (no name), pick (Target), num (Noise), type (Zero variance feature).

In [4]:
cols_to_drop = ['Unnamed: 65', 'pick', 'num', 'type']
df_09_21 = df_09_21.drop(columns=cols_to_drop)

In [5]:
df_09_21.head(10)

,player_name,team,conf,GP,Min_per,Ortg,usg,eFG,TS_per,ORB_per,...,ogbpm,dgbpm,oreb,dreb,treb,ast,stl,blk,pts,role
0,DeAndrae Ross,South Alabama,SB,26,29.5,97.3,16.6,42.5,44.43,1.6,...,-2.781990,-1.941150,0.1923,0.6154,0.8077,1.1923,0.3462,0.0385,3.8846,NaN
1,Pooh Williams,Utah St.,WAC,34,60.9,108.3,14.9,52.4,54.48,3.8,...,-0.052263,-0.247934,0.6765,1.2647,1.9412,1.8235,0.4118,0.2353,5.9412,NaN
2,Jesus Verdejo,South Florida,BE,27,72.0,96.2,21.8,45.7,47.98,2.1,...,1.548230,-0.883163,0.6296,2.3333,2.9630,1.9630,0.4815,0.0000,12.1852,NaN
3,Mike Hornbuckle,Pepperdine,WCC,30,44.5,97.7,16.0,53.6,53.69,4.1,...,-0.342775,-0.393459,0.7000,1.4333,2.1333,1.1000,0.5667,0.1333,4.9333,NaN
4,Anthony Brown,Pacific,BW,33,56.2,96.5,22.0,52.8,54.31,8.3,...,-1.684860,-0.668318,1.4242,3.3030,4.7273,0.8485,0.4545,0.3333,7.5758,NaN
5,Nick Rodgers,Butler,Horz,6,0.7,0.0,0.0,0.0,0.00,0.0,...,-0.956800,3.744910,0.0000,0.3333,0.3333,0.3333,0.0000,0.0000,0.0000,NaN
6,Dana Smith,Longwood,ind,27,77.8,104.8,23.0,53.4,56.30,6.8,...,1.103250,-0.908391,2.1481,3.8889,6.0370,1.8148,1.7778,0.7407,15.2593,NaN
7,Matt Beck,Fordham,A10,19,10.4,131.9,3.3,90.0,90.00,2.6,...,-3.490760,-2.086900,0.1579,0.1579,0.3158,0.1053,0.1053,0.0000,0.4737,NaN
8,Justin Drummond,Wagner,NEC,30,82.8,99.7,20.5,48.8,53.07,2.9,...,0.250984,0.919272,0.8000,3.9333,4.7333,4.1333,1.7333,0.8000,10.5000,NaN
9,Jamal Smith,Wagner,NEC,30,80.4,92.5,23.0,43.5,45.29,6.4,...,-1.892540,-1.913070,1.7333,3.4667,5.2000,1.7333,1.2333,0.1000,11.8667,NaN


## Remove duplicate name, only retain the data for the last year of a player.

In [6]:
df_09_21 = df_09_21.sort_values(by='year')

df_09_21 = df_09_21.drop_duplicates(subset=['pid'], keep='last')

df_09_21 = df_09_21.reset_index(drop=True)

In [7]:
df_09_21.head(10)

,player_name,team,conf,GP,Min_per,Ortg,usg,eFG,TS_per,ORB_per,...,ogbpm,dgbpm,oreb,dreb,treb,ast,stl,blk,pts,role
0,Jesus Verdejo,South Florida,BE,27,72.0,96.2,21.8,45.7,47.98,2.1,...,1.548230,-0.883163,0.6296,2.3333,2.9630,1.9630,0.4815,0.0000,12.1852,NaN
1,Mike Hornbuckle,Pepperdine,WCC,30,44.5,97.7,16.0,53.6,53.69,4.1,...,-0.342775,-0.393459,0.7000,1.4333,2.1333,1.1000,0.5667,0.1333,4.9333,NaN
2,Anthony Brown,Pacific,BW,33,56.2,96.5,22.0,52.8,54.31,8.3,...,-1.684860,-0.668318,1.4242,3.3030,4.7273,0.8485,0.4545,0.3333,7.5758,NaN
3,Justin Drummond,Wagner,NEC,30,82.8,99.7,20.5,48.8,53.07,2.9,...,0.250984,0.919272,0.8000,3.9333,4.7333,4.1333,1.7333,0.8000,10.5000,NaN
4,Jamal Smith,Wagner,NEC,30,80.4,92.5,23.0,43.5,45.29,6.4,...,-1.892540,-1.913070,1.7333,3.4667,5.2000,1.7333,1.2333,0.1000,11.8667,NaN
5,Tyrell Biggs,Pittsburgh,BE,34,57.9,113.9,13.7,55.3,56.37,9.8,...,1.965260,1.557650,2.0000,2.2941,4.2941,0.7059,0.4118,0.4706,6.4706,NaN
6,Simon Harris,North Carolina St.,ACC,19,18.4,95.1,8.2,45.2,48.39,6.9,...,-2.790750,0.388021,0.6316,1.3684,2.0000,0.2632,0.1579,0.1579,1.2632,NaN
7,Gary Wilkinson,Utah St.,WAC,34,79.9,124.8,24.0,57.9,63.97,7.2,...,5.596360,-0.233409,1.6765,4.9706,6.6471,1.2353,0.4706,0.3824,16.8824,NaN
8,Tawrence Walton,Chicago St.,ind,30,67.3,102.1,15.1,49.7,54.46,9.4,...,-1.266710,-2.315280,2.5000,3.5000,6.0000,1.2333,0.9333,0.5000,7.4333,NaN
9,Garrett Temple,LSU,SEC,35,75.2,104.4,15.9,43.1,49.61,4.8,...,0.740255,3.121290,1.3143,3.2286,4.5429,3.8286,1.7143,0.6857,7.0571,NaN


## Merge two dataset and create target

In [8]:
df_draft_09_21 = pd.read_excel('DraftedPlayers2009-2021.xlsx')

In [9]:
df_draft_09_21.head(10)

,PLAYER,TEAM,AFFILIATION,YEAR,ROUND,ROUND.1,OVERALL
0,NaN,NaN,NaN,NaN,NUMBER,PICK,PICK
1,Cade Cunningham,Detroit Pistons,Oklahoma State,2021.0,1,1,1
2,Jalen Green,Houston Rockets,Ignite (G League),2021.0,1,2,2
3,Evan Mobley,Cleveland Cavaliers,Southern California,2021.0,1,3,3
4,Scottie Barnes,Toronto Raptors,Florida State,2021.0,1,4,4
5,Jalen Suggs,Orlando Magic,Gonzaga,2021.0,1,5,5
6,Josh Giddey,Oklahoma City Thunder,NBA Global Academy (Australia),2021.0,1,6,6
7,Jonathan Kuminga,Golden State Warriors,Ignite (G League),2021.0,1,7,7
8,Franz Wagner,Orlando Magic,Michigan,2021.0,1,8,8
9,Davion Mitchell,Sacramento Kings,Baylor,2021.0,1,9,9


In [10]:
# Drop index 0

df_draft_09_21 = df_draft_09_21.drop(0)
df_draft_09_21 = df_draft_09_21.reset_index(drop=True)

In [11]:
df_draft_09_21.head(10)

,PLAYER,TEAM,AFFILIATION,YEAR,ROUND,ROUND.1,OVERALL
0,Cade Cunningham,Detroit Pistons,Oklahoma State,2021.0,1,1,1
1,Jalen Green,Houston Rockets,Ignite (G League),2021.0,1,2,2
2,Evan Mobley,Cleveland Cavaliers,Southern California,2021.0,1,3,3
3,Scottie Barnes,Toronto Raptors,Florida State,2021.0,1,4,4
4,Jalen Suggs,Orlando Magic,Gonzaga,2021.0,1,5,5
5,Josh Giddey,Oklahoma City Thunder,NBA Global Academy (Australia),2021.0,1,6,6
6,Jonathan Kuminga,Golden State Warriors,Ignite (G League),2021.0,1,7,7
7,Franz Wagner,Orlando Magic,Michigan,2021.0,1,8,8
8,Davion Mitchell,Sacramento Kings,Baylor,2021.0,1,9,9
9,Ziaire Williams,New Orleans Pelicans,Stanford,2021.0,1,10,10


In [12]:
df_merged = df_09_21.merge(
    df_draft_09_21[['PLAYER', 'YEAR', 'ROUND']], 
    left_on=['player_name', 'year'], 
    right_on=['PLAYER', 'YEAR'], 
    how='left'
)

In [13]:
df_merged.head(10)

,player_name,team,conf,GP,Min_per,Ortg,usg,eFG,TS_per,ORB_per,...,dreb,treb,ast,stl,blk,pts,role,PLAYER,YEAR,ROUND
0,Jesus Verdejo,South Florida,BE,27,72.0,96.2,21.8,45.7,47.98,2.1,...,2.3333,2.9630,1.9630,0.4815,0.0000,12.1852,NaN,NaN,NaN,NaN
1,Mike Hornbuckle,Pepperdine,WCC,30,44.5,97.7,16.0,53.6,53.69,4.1,...,1.4333,2.1333,1.1000,0.5667,0.1333,4.9333,NaN,NaN,NaN,NaN
2,Anthony Brown,Pacific,BW,33,56.2,96.5,22.0,52.8,54.31,8.3,...,3.3030,4.7273,0.8485,0.4545,0.3333,7.5758,NaN,NaN,NaN,NaN
3,Justin Drummond,Wagner,NEC,30,82.8,99.7,20.5,48.8,53.07,2.9,...,3.9333,4.7333,4.1333,1.7333,0.8000,10.5000,NaN,NaN,NaN,NaN
4,Jamal Smith,Wagner,NEC,30,80.4,92.5,23.0,43.5,45.29,6.4,...,3.4667,5.2000,1.7333,1.2333,0.1000,11.8667,NaN,NaN,NaN,NaN
5,Tyrell Biggs,Pittsburgh,BE,34,57.9,113.9,13.7,55.3,56.37,9.8,...,2.2941,4.2941,0.7059,0.4118,0.4706,6.4706,NaN,NaN,NaN,NaN
6,Simon Harris,North Carolina St.,ACC,19,18.4,95.1,8.2,45.2,48.39,6.9,...,1.3684,2.0000,0.2632,0.1579,0.1579,1.2632,NaN,NaN,NaN,NaN
7,Gary Wilkinson,Utah St.,WAC,34,79.9,124.8,24.0,57.9,63.97,7.2,...,4.9706,6.6471,1.2353,0.4706,0.3824,16.8824,NaN,NaN,NaN,NaN
8,Tawrence Walton,Chicago St.,ind,30,67.3,102.1,15.1,49.7,54.46,9.4,...,3.5000,6.0000,1.2333,0.9333,0.5000,7.4333,NaN,NaN,NaN,NaN
9,Garrett Temple,LSU,SEC,35,75.2,104.4,15.9,43.1,49.61,4.8,...,3.2286,4.5429,3.8286,1.7143,0.6857,7.0571,NaN,NaN,NaN,NaN


In [14]:
def assign_draft_status(row):
    if pd.isna(row['ROUND']):
        return 0 
    elif row['ROUND'] == 1:
        return 1
    elif row['ROUND'] == 2:
        return 2
    else:
        return 0

In [15]:
df_merged['draft_status'] = df_merged.apply(assign_draft_status, axis=1)

In [16]:
df_merged = df_merged.drop(columns=['PLAYER', 'YEAR', 'ROUND'])

In [17]:
df_merged.head(10)

,player_name,team,conf,GP,Min_per,Ortg,usg,eFG,TS_per,ORB_per,...,dgbpm,oreb,dreb,treb,ast,stl,blk,pts,role,draft_status
0,Jesus Verdejo,South Florida,BE,27,72.0,96.2,21.8,45.7,47.98,2.1,...,-0.883163,0.6296,2.3333,2.9630,1.9630,0.4815,0.0000,12.1852,NaN,0
1,Mike Hornbuckle,Pepperdine,WCC,30,44.5,97.7,16.0,53.6,53.69,4.1,...,-0.393459,0.7000,1.4333,2.1333,1.1000,0.5667,0.1333,4.9333,NaN,0
2,Anthony Brown,Pacific,BW,33,56.2,96.5,22.0,52.8,54.31,8.3,...,-0.668318,1.4242,3.3030,4.7273,0.8485,0.4545,0.3333,7.5758,NaN,0
3,Justin Drummond,Wagner,NEC,30,82.8,99.7,20.5,48.8,53.07,2.9,...,0.919272,0.8000,3.9333,4.7333,4.1333,1.7333,0.8000,10.5000,NaN,0
4,Jamal Smith,Wagner,NEC,30,80.4,92.5,23.0,43.5,45.29,6.4,...,-1.913070,1.7333,3.4667,5.2000,1.7333,1.2333,0.1000,11.8667,NaN,0
5,Tyrell Biggs,Pittsburgh,BE,34,57.9,113.9,13.7,55.3,56.37,9.8,...,1.557650,2.0000,2.2941,4.2941,0.7059,0.4118,0.4706,6.4706,NaN,0
6,Simon Harris,North Carolina St.,ACC,19,18.4,95.1,8.2,45.2,48.39,6.9,...,0.388021,0.6316,1.3684,2.0000,0.2632,0.1579,0.1579,1.2632,NaN,0
7,Gary Wilkinson,Utah St.,WAC,34,79.9,124.8,24.0,57.9,63.97,7.2,...,-0.233409,1.6765,4.9706,6.6471,1.2353,0.4706,0.3824,16.8824,NaN,0
8,Tawrence Walton,Chicago St.,ind,30,67.3,102.1,15.1,49.7,54.46,9.4,...,-2.315280,2.5000,3.5000,6.0000,1.2333,0.9333,0.5000,7.4333,NaN,0
9,Garrett Temple,LSU,SEC,35,75.2,104.4,15.9,43.1,49.61,4.8,...,3.121290,1.3143,3.2286,4.5429,3.8286,1.7143,0.6857,7.0571,NaN,0


In [18]:
print(df_merged['draft_status'].value_counts())

draft_status
0    25152
1      313
2      280
Name: count, dtype: int64


## Fix `yr` and `ht`

In [19]:
# yr

yr_map = {'Fr': 1, 'So': 2, 'Jr': 3, 'Sr': 4}

df_merged['yr_num'] = df_merged['yr'].map(yr_map)

df_merged = df_merged.drop(columns=['yr'])

In [20]:
# ht

def parse_ht(ht_str):
    if pd.isna(ht_str):
        return np.nan
    ht_str = str(ht_str).strip()
    
    month_to_feet = {'Apr': 4, 'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8}
    
    # ('2-Jun' -> 6 * 12 + 2 = 74 inches)
    m1 = re.match(r'^(\d{1,2})\-(Apr|May|Jun|Jul|Aug)$', ht_str)
    if m1: 
        return month_to_feet[m1.group(2)] * 12 + int(m1.group(1))
        
    # ('Jun-00' -> 6 * 12 + 0 = 72 inches)
    m2 = re.match(r'^(Apr|May|Jun|Jul|Aug)\-(\d{1,2})$', ht_str)
    if m2: 
        return month_to_feet[m2.group(1)] * 12 + int(m2.group(2))
    
    # ('6-2' -> 6 * 12 + 2 = 74 inches)
    m3 = re.match(r'^(\d)\-(\d{1,2})$', ht_str)
    if m3: 
        return int(m3.group(1)) * 12 + int(m3.group(2))
    
    # ("6'4" -> 6 * 12 + 4 = 76 inches)
    m4 = re.match(r'^(\d)\'(\d{1,2})$', ht_str)
    if m4: 
        return int(m4.group(1)) * 12 + int(m4.group(2))
    
    return np.nan

In [21]:
df_merged['ht_inches'] = df_merged['ht'].apply(parse_ht)
df_merged = df_merged.drop(columns=['ht'])

In [22]:
# Drop players with missing height data
df_merged = df_merged.dropna(subset=['ht_inches'])

In [23]:
df_merged.head(10)

,player_name,team,conf,GP,Min_per,Ortg,usg,eFG,TS_per,ORB_per,...,dreb,treb,ast,stl,blk,pts,role,draft_status,yr_num,ht_inches
0,Jesus Verdejo,South Florida,BE,27,72.0,96.2,21.8,45.7,47.98,2.1,...,2.3333,2.9630,1.9630,0.4815,0.0000,12.1852,NaN,0,4.0,76.0
1,Mike Hornbuckle,Pepperdine,WCC,30,44.5,97.7,16.0,53.6,53.69,4.1,...,1.4333,2.1333,1.1000,0.5667,0.1333,4.9333,NaN,0,4.0,76.0
2,Anthony Brown,Pacific,BW,33,56.2,96.5,22.0,52.8,54.31,8.3,...,3.3030,4.7273,0.8485,0.4545,0.3333,7.5758,NaN,0,4.0,80.0
3,Justin Drummond,Wagner,NEC,30,82.8,99.7,20.5,48.8,53.07,2.9,...,3.9333,4.7333,4.1333,1.7333,0.8000,10.5000,NaN,0,4.0,78.0
4,Jamal Smith,Wagner,NEC,30,80.4,92.5,23.0,43.5,45.29,6.4,...,3.4667,5.2000,1.7333,1.2333,0.1000,11.8667,NaN,0,4.0,77.0
5,Tyrell Biggs,Pittsburgh,BE,34,57.9,113.9,13.7,55.3,56.37,9.8,...,2.2941,4.2941,0.7059,0.4118,0.4706,6.4706,NaN,0,4.0,80.0
6,Simon Harris,North Carolina St.,ACC,19,18.4,95.1,8.2,45.2,48.39,6.9,...,1.3684,2.0000,0.2632,0.1579,0.1579,1.2632,NaN,0,4.0,77.0
7,Gary Wilkinson,Utah St.,WAC,34,79.9,124.8,24.0,57.9,63.97,7.2,...,4.9706,6.6471,1.2353,0.4706,0.3824,16.8824,NaN,0,4.0,81.0
8,Tawrence Walton,Chicago St.,ind,30,67.3,102.1,15.1,49.7,54.46,9.4,...,3.5000,6.0000,1.2333,0.9333,0.5000,7.4333,NaN,0,4.0,77.0
9,Garrett Temple,LSU,SEC,35,75.2,104.4,15.9,43.1,49.61,4.8,...,3.2286,4.5429,3.8286,1.7143,0.6857,7.0571,NaN,0,4.0,78.0


## Missing value inputation

In [24]:
missing_counts = df_merged.isnull().sum()

missing_only = missing_counts[missing_counts > 0].sort_values(ascending=False)

print(missing_only)

Rec Rank                           18876
dunksmade/(dunksmade+dunksmiss)    14580
midmade/(midmade+midmiss)           4619
rimmade/(rimmade+rimmiss)           4558
rimmade+rimmiss                     2528
midmade                             2528
rimmade                             2528
dunksmade                           2528
midmade+midmiss                     2528
dunksmiss+dunksmade                 2528
ast/tov                             2380
role                                1624
yr_num                                23
drtg                                   3
adrtg                                  3
stops                                  3
dporpag                                3
bpm                                    3
obpm                                   3
ogbpm                                  3
dgbpm                                  3
dbpm                                   3
gbpm                                   3
oreb                                   1
mp              

In [25]:
df_merged.head(10)

,player_name,team,conf,GP,Min_per,Ortg,usg,eFG,TS_per,ORB_per,...,dreb,treb,ast,stl,blk,pts,role,draft_status,yr_num,ht_inches
0,Jesus Verdejo,South Florida,BE,27,72.0,96.2,21.8,45.7,47.98,2.1,...,2.3333,2.9630,1.9630,0.4815,0.0000,12.1852,NaN,0,4.0,76.0
1,Mike Hornbuckle,Pepperdine,WCC,30,44.5,97.7,16.0,53.6,53.69,4.1,...,1.4333,2.1333,1.1000,0.5667,0.1333,4.9333,NaN,0,4.0,76.0
2,Anthony Brown,Pacific,BW,33,56.2,96.5,22.0,52.8,54.31,8.3,...,3.3030,4.7273,0.8485,0.4545,0.3333,7.5758,NaN,0,4.0,80.0
3,Justin Drummond,Wagner,NEC,30,82.8,99.7,20.5,48.8,53.07,2.9,...,3.9333,4.7333,4.1333,1.7333,0.8000,10.5000,NaN,0,4.0,78.0
4,Jamal Smith,Wagner,NEC,30,80.4,92.5,23.0,43.5,45.29,6.4,...,3.4667,5.2000,1.7333,1.2333,0.1000,11.8667,NaN,0,4.0,77.0
5,Tyrell Biggs,Pittsburgh,BE,34,57.9,113.9,13.7,55.3,56.37,9.8,...,2.2941,4.2941,0.7059,0.4118,0.4706,6.4706,NaN,0,4.0,80.0
6,Simon Harris,North Carolina St.,ACC,19,18.4,95.1,8.2,45.2,48.39,6.9,...,1.3684,2.0000,0.2632,0.1579,0.1579,1.2632,NaN,0,4.0,77.0
7,Gary Wilkinson,Utah St.,WAC,34,79.9,124.8,24.0,57.9,63.97,7.2,...,4.9706,6.6471,1.2353,0.4706,0.3824,16.8824,NaN,0,4.0,81.0
8,Tawrence Walton,Chicago St.,ind,30,67.3,102.1,15.1,49.7,54.46,9.4,...,3.5000,6.0000,1.2333,0.9333,0.5000,7.4333,NaN,0,4.0,77.0
9,Garrett Temple,LSU,SEC,35,75.2,104.4,15.9,43.1,49.61,4.8,...,3.2286,4.5429,3.8286,1.7143,0.6857,7.0571,NaN,0,4.0,78.0


In [26]:
# 1. 'Rec Rank': fill 999 means "not ranked"
df_merged['Rec Rank'] = df_merged['Rec Rank'].fillna(999)

In [27]:
# 2. 'dunksmade/(dunksmade+dunksmiss)': drop this feature since this is a derived feature and has many missing values
df_merged = df_merged.drop(columns=['dunksmade/(dunksmade+dunksmiss)'])

In [28]:
# 3. 'role': fill 'Unknown' for missing role since role is an important feature and we don't want to drop players with missing role
df_merged['role'] = df_merged['role'].fillna('Unknown')

In [29]:
# Create ht_group
bins_ht = [0, 75, 79, 100] 
labels_ht = ['Guard_Size', 'Wing_Size', 'Big_Size']
df_merged['ht_group'] = pd.cut(df_merged['ht_inches'], bins=bins_ht, labels=labels_ht, include_lowest=True)

In [30]:
# 4. 'rimmade', 'rimmade+rimmiss', 'midmade', 'midmade+midmiss', 'rimmade/(rimmade+rimmiss)', 'midmade/(midmade+midmiss)'
# Based on ht_group, use the actual total number of two-point attempts (twoPM/twoPA) to allocate players' rim-made and mid-range made shots
df_merged['temp_rim_ratio_pm'] = df_merged['rimmade'] / df_merged['twoPM'].replace(0, np.nan)
df_merged['temp_rim_ratio_pa'] = df_merged['rimmade+rimmiss'] / df_merged['twoPA'].replace(0, np.nan)

group_rim_pm = df_merged.groupby('ht_group')['temp_rim_ratio_pm'].transform('median')
group_rim_pa = df_merged.groupby('ht_group')['temp_rim_ratio_pa'].transform('median')

missing_idx = df_merged['rimmade'].isnull()

df_merged.loc[missing_idx, 'rimmade'] = df_merged.loc[missing_idx, 'twoPM'] * group_rim_pm[missing_idx]
df_merged.loc[missing_idx, 'midmade'] = df_merged.loc[missing_idx, 'twoPM'] * (1 - group_rim_pm[missing_idx])

df_merged.loc[missing_idx, 'rimmade+rimmiss'] = df_merged.loc[missing_idx, 'twoPA'] * group_rim_pa[missing_idx]
df_merged.loc[missing_idx, 'midmade+midmiss'] = df_merged.loc[missing_idx, 'twoPA'] * (1 - group_rim_pa[missing_idx])

df_merged['rimmade/(rimmade+rimmiss)'] = df_merged['rimmade'] / df_merged['rimmade+rimmiss'].replace(0, np.nan)
df_merged['midmade/(midmade+midmiss)'] = df_merged['midmade'] / df_merged['midmade+midmiss'].replace(0, np.nan)
df_merged['rimmade/(rimmade+rimmiss)'] = df_merged['rimmade/(rimmade+rimmiss)'].fillna(0)
df_merged['midmade/(midmade+midmiss)'] = df_merged['midmade/(midmade+midmiss)'].fillna(0)

C:\Users\賴品瑄\AppData\Local\Temp\ipykernel_8260\4262866767.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_rim_pm = df_merged.groupby('ht_group')['temp_rim_ratio_pm'].transform('median')
C:\Users\賴品瑄\AppData\Local\Temp\ipykernel_8260\4262866767.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_rim_pa = df_merged.groupby('ht_group')['temp_rim_ratio_pa'].transform('median')


In [31]:
# 5. 'dunksmade', 'dunksmiss+dunksmade': fill 0 for missing values since if a player has missing dunksmade or dunksmiss+dunksmade, it likely means the player has 0 dunks made and 0 dunks attempted
df_merged['dunksmade'] = df_merged['dunksmade'].fillna(0)
df_merged['dunksmiss+dunksmade'] = df_merged['dunksmiss+dunksmade'].fillna(0)

In [32]:
# 6. 'yr_num': fill missng values with median
df_merged['yr_num'] = df_merged['yr_num'].fillna(df_merged['yr_num'].median())

In [33]:
# 7. 'ast/tov': fill 0 for missing values since if a player has missing ast/tov, it likely means the player has 0 assists and 0 turnovers
df_merged['ast/tov'] = df_merged['ast/tov'].fillna(0)

In [34]:
temp_cols = ['ht_group', 'temp_rim_ratio_pm', 'temp_rim_ratio_pa']
df_merged = df_merged.drop(columns=[c for c in temp_cols if c in df_merged.columns])

In [35]:
# 8. In the end, there will only be three entries with missing traditional and advanced stats, and they’re all undrafted, so I’ll just drop them.
df_merged = df_merged.dropna()

In [36]:
missing_counts_after = df_merged.isnull().sum()

missing_only_after = missing_counts_after[missing_counts_after > 0].sort_values(ascending=False)

print(missing_only_after)

Series([], dtype: int64)


In [37]:
df_merged.head(10)

,player_name,team,conf,GP,Min_per,Ortg,usg,eFG,TS_per,ORB_per,...,dreb,treb,ast,stl,blk,pts,role,draft_status,yr_num,ht_inches
0,Jesus Verdejo,South Florida,BE,27,72.0,96.2,21.8,45.7,47.98,2.1,...,2.3333,2.9630,1.9630,0.4815,0.0000,12.1852,Unknown,0,4.0,76.0
1,Mike Hornbuckle,Pepperdine,WCC,30,44.5,97.7,16.0,53.6,53.69,4.1,...,1.4333,2.1333,1.1000,0.5667,0.1333,4.9333,Unknown,0,4.0,76.0
2,Anthony Brown,Pacific,BW,33,56.2,96.5,22.0,52.8,54.31,8.3,...,3.3030,4.7273,0.8485,0.4545,0.3333,7.5758,Unknown,0,4.0,80.0
3,Justin Drummond,Wagner,NEC,30,82.8,99.7,20.5,48.8,53.07,2.9,...,3.9333,4.7333,4.1333,1.7333,0.8000,10.5000,Unknown,0,4.0,78.0
4,Jamal Smith,Wagner,NEC,30,80.4,92.5,23.0,43.5,45.29,6.4,...,3.4667,5.2000,1.7333,1.2333,0.1000,11.8667,Unknown,0,4.0,77.0
5,Tyrell Biggs,Pittsburgh,BE,34,57.9,113.9,13.7,55.3,56.37,9.8,...,2.2941,4.2941,0.7059,0.4118,0.4706,6.4706,Unknown,0,4.0,80.0
6,Simon Harris,North Carolina St.,ACC,19,18.4,95.1,8.2,45.2,48.39,6.9,...,1.3684,2.0000,0.2632,0.1579,0.1579,1.2632,Unknown,0,4.0,77.0
7,Gary Wilkinson,Utah St.,WAC,34,79.9,124.8,24.0,57.9,63.97,7.2,...,4.9706,6.6471,1.2353,0.4706,0.3824,16.8824,Unknown,0,4.0,81.0
8,Tawrence Walton,Chicago St.,ind,30,67.3,102.1,15.1,49.7,54.46,9.4,...,3.5000,6.0000,1.2333,0.9333,0.5000,7.4333,Unknown,0,4.0,77.0
9,Garrett Temple,LSU,SEC,35,75.2,104.4,15.9,43.1,49.61,4.8,...,3.2286,4.5429,3.8286,1.7143,0.6857,7.0571,Unknown,0,4.0,78.0


## Basic Feature Engineering

### 1. Offensive playmaker who handles the ball well and is an excellent passer

In [38]:
df_merged['Playmaker_Usg'] = df_merged['usg'] * df_merged['AST_per']

### 2. A high-scoring, foul-drawing disruptor

df_merged['Scoring_Gravity'] = df_merged['pts'] * df_merged['ftr']

### 3. Selflessness Index: How many assists does player generate per possession? If this number is high, it indicates that this player is a highly pure point guard.

In [39]:
df_merged['AST_to_USG'] = df_merged['AST_per'] / (df_merged['usg'] + 0.001)

### 4. Is the player ranked among the top 25 high school players in the nation? Convert to 1 and 0

In [40]:
df_merged['Elite_Recruit'] = (df_merged['Rec Rank'] <= 25).astype(int)

### 5. Players who can put up impressive stats as freshmen are the darlings of NBA scouts—commonly known as “one-and-done” players.

In [41]:
df_merged['Freshman_Star'] = ((df_merged['yr_num'] == 1) & (df_merged['bpm'] > 5)).astype(int)

## Train, validation, test split

In [42]:
df_train = df_merged[df_merged['year'] <= 2018].copy()
df_val = df_merged[(df_merged['year'] == 2019) | (df_merged['year'] == 2020)].copy()
df_test = df_merged[df_merged['year'] == 2021].copy()

df_train.to_csv('NBA_Train.csv', index=False)
df_val.to_csv('NBA_Validation.csv', index=False)
df_test.to_csv('NBA_Test.csv', index=False)